**[Source]** Donghwan Project (최종 학습 단계) + Jisoo Project 9단계(pko-T5 전체 학습) 결과 재사용
**[Status]** ADAPTED
**[Role]** 08번에서 채택된 모델의 최종 Train 학습. 07번에서 재사용이 검증(REUSE_OK)되면 새로 학습하지 않고 기존 체크포인트·학습 기록을 그대로 사용
**[Modification]** 동환의 학습 루프를 실행하는 대신 지수의 실제 학습 결과를 재사용한다(중복 학습 방지). ET5 전체 학습은 사용자 확인 없이 시작하지 않도록 잠가 두었다.
**Test는 열지 않는다.**

# 09. 최종 학습 (재사용 우선)
- 08번에서 채택된 모델이 pko-T5이고 07번 재사용 판정이 REUSE_OK이면 **재학습하지 않는다**. 재학습하면 같은 설정에서도 GPU 비결정성 때문에 결과가 미세하게 달라져 이미 검증된 예측과 어긋날 수 있다.
- 학습 기록(loss 곡선, 학습 시간, update 수, 환경)을 그대로 보고한다.
- ET5가 08번에서 이기는 경우에만 전체 학습이 필요하며, 학습 시간이 매우 길어(동환 속도 기준 수십 시간 추정) 사용자 확인이 있을 때만 실행하도록 잠겨 있다.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 08번 판정과 07번 재사용 판정 확인
M8 = json.loads((P.RUNS / "model_comparison_08.json").read_text(encoding="utf-8"))
RU = json.loads((P.RUNS / "reuse_verification_pkot5.json").read_text(encoding="utf-8"))
sel = M8["selection"]["winner_primary"]; print("08번 주 지표 승자:", sel, "| ET5 상태:", M8["et5_status"].get("run"), M8["et5_status"].get("reason"))
print("07번 재사용 판정:", "REUSE_OK" if RU["reuse_ok"] else "재사용 불가", "| 체크포인트 SHA256:", RU["checkpoint_weight_sha256"][:16] + "…")
assert sel == "pko-T5" and RU["reuse_ok"], "pko-T5 재사용 경로가 아닙니다 → 이 노트북의 재학습 셀(셀 3)을 사용자 확인 후 실행해야 함"
ROUTE = "REUSE_PKOT5_FULL_1EPOCH"; print("경로:", ROUTE)

08번 주 지표 승자: pko-T5 | ET5 상태: False RUN_ET5=False (기본값)
07번 재사용 판정: REUSE_OK | 체크포인트 SHA256: ed95a6fc64cd3902…
경로: REUSE_PKOT5_FULL_1EPOCH


In [3]:
# [셀 2] 재사용하는 학습의 실제 기록
FR = json.loads((P.J_OUT / "pkot5_full" / "pkot5_full_results.json").read_text(encoding="utf-8"))
TS = FR["train_summary"]; print(pd.Series({k: TS[k] for k in TS if not isinstance(TS[k], (dict, list))}).to_string())
LOG = pd.read_csv(P.J_OUT / "pkot5_full" / "train_log.csv")
print("\n로그 행 수:", len(LOG), "| 마지막 step:", int(LOG.step.iloc[-1]), "| 마지막 train loss:", round(float(LOG.train_loss.iloc[-1]), 4), "| 학습 경과(h):", round(float(LOG.elapsed_sec.iloc[-1]) / 3600, 2))
print("※ 이 실행은 중간에 재개(resumed_from_checkpoint=%s)되었으므로 elapsed_sec은 누적값이다." % TS.get("resumed_from_checkpoint"))
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.5, 3.6)); ax.plot(LOG.step, LOG.train_loss, lw=1); ax.set_yscale("log"); ax.set_xlabel("Update step"); ax.set_ylabel("Train loss (log scale)"); ax.set_title("pko-T5 full-train loss (1 epoch)")
plt.tight_layout(); plt.savefig(P.REPORTS / "fig09_train_loss.png", dpi=150); plt.close()
print("저장: reports/fig09_train_loss.png")
FINAL_TRAIN = {"route": ROUTE, "model": "paust/pko-t5-base", "checkpoint_sha256": RU["checkpoint_weight_sha256"], "updates": TS["updates_done"], "final_train_loss": TS["final_train_loss"], "gpu": TS.get("gpu"), "torch": TS.get("torch"), "transformers": TS.get("transformers")}
(P.RUNS / "final_training_09.json").write_text(json.dumps(FINAL_TRAIN, ensure_ascii=False, indent=2), encoding="utf-8"); print(FINAL_TRAIN)

updates_done                                                 61670
n_updates_planned                                            61670
micro_batches_this_session                                   53670
nan_or_inf                                                   False
train_time_sec_total                                       27677.9
train_time_measured                                           True
resumed_from_checkpoint                                       True
final_train_loss                                          0.130688
peak_gpu_mem_gb_this_session                                  6.65
precision                                                     bf16
gpu                             NVIDIA GeForce RTX 5060 Laptop GPU
gpu_state_start                                               None
gpu_state_end                                                 None
weight_decay_groups                   HF 관례(bias/LayerNorm 제외)
torch                                                  2.7.1+cu128

In [4]:
# [셀 3] (잠김) ET5 전체 학습 — 08번에서 ET5가 이기고, 사용자가 시간 비용을 확인했을 때만 True로 바꾼다
ET5_FULL_CONFIRMED_BY_USER = False
if not ET5_FULL_CONFIRMED_BY_USER:
    print("ET5 전체 학습: 실행하지 않음(잠김). ET5 결과·시간에 대한 결론 없음.")
else:
    import seq2seq_tools as S2S
    from transformers import AutoTokenizer
    assert P.ET5_DIR and Path(P.ET5_DIR).exists(), "ET5 가중치 경로 필요"
    et5 = M8["et5_status"]; assert et5.get("run"), "08번 ET5 screening 결과가 있어야 함"
    tok = AutoTokenizer.from_pretrained(P.ET5_DIR); PREFIX = "맞춤법 교정: "
    TRN = common.read_split(P, "train", columns=["input", "target"])
    ex, info = S2S.build_examples(tok, [r["input"] for r in TRN], [r["target"] for r in TRN], *et5["max_length"], prefix=PREFIX)
    model, s = S2S.train(P.ET5_DIR, tok, ex, P.MODELS / "et5_full", lr=et5["lr_best"], micro=8, accum=2, seed=SEED); print(s)

ET5 전체 학습: 실행하지 않음(잠김). ET5 결과·시간에 대한 결론 없음.


## 해석
- **경로 결정: REUSE_PKOT5_FULL_1EPOCH.** 08번 주 지표 승자가 pko-T5이고 07번 재사용 판정이 REUSE_OK이므로 새로 학습하지 않았다. 이 노트북이 새로 만든 학습 결과는 없다(loss 곡선 그림과 요약 파일 정리만 수행).
- 재사용한 학습의 실제 기록: update 61,670회(= 계획과 동일), 최종 train loss 0.131(로그 마지막 평균 0.128), NaN/inf 없음, bf16, RTX 5060 Laptop GPU, torch 2.7.1+cu128, transformers 4.44.2, 누적 학습 시간 27,678초(약 7.7시간).
- **주의**: 이 학습은 중간에 체크포인트에서 재개되었다(`resumed_from_checkpoint=True`). 재개 학습이 단번에 학습한 것과 완전히 같은 결과를 보장하는지는 이번에 검증하지 않았다(설정과 데이터 순서가 같다는 점은 지수 9번 기록에 근거).
- train loss는 학습 곡선 확인용이며 일반화 성능의 증거가 아니다 → 성능은 11번 Validation 평가에서만 판단한다.
- ET5 전체 학습은 잠겨 있어 **실행하지 않았다**. ET5에 대한 결론 없음.